# Unscented Kalman Filter on AIS Data
This notebook loads AIS data (first 1000 samples), runs a UKF, logs innovations and NIS, and produces plots so you can compare with KF/EKF.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Load AIS Data (first 1000 points)

In [ ]:
df = pd.read_csv("Data/AIS_2016_11_22.csv")

df["BaseDateTime"] = pd.to_datetime(df["BaseDateTime"])
df = df.sort_values("BaseDateTime").reset_index(drop=True)

# limit to first 1000 AIS points
df = df.iloc[:1000]

lat = df["LAT"].values
lon = df["LON"].values

positions = np.column_stack((lat, lon))

times = df["BaseDateTime"].astype("int64") // 10**9
times = times.to_numpy()

dt = np.diff(times, prepend=times[0])

print("Using", len(positions), "AIS points")
print("dt range:", dt.min(), "to", dt.max())

## Sigma Point Generator

In [ ]:
def sigma_points(x, P, alpha=1e-3, beta=2, kappa=0):

    n = len(x)
    lambda_ = alpha**2 * (n + kappa) - n

    sigma = np.zeros((2*n+1, n))
    sigma[0] = x

    sqrt_matrix = np.linalg.cholesky((n + lambda_) * P)

    for i in range(n):
        sigma[i+1] = x + sqrt_matrix[:, i]
        sigma[n+i+1] = x - sqrt_matrix[:, i]

    return sigma, lambda_

## Run UKF and Log Innovations

In [ ]:
x = np.array([positions[0,0],0.0,positions[0,1],0.0])

P = np.eye(4) * 10

Q = np.eye(4) * 0.1
R = np.eye(2) * 1e-8

n = 4
alpha = 1e-3
beta = 2
kappa = 0

ukf_estimates = []
innovations = []
nis_values = []

for k in range(len(positions)):

    dt_k = dt[k]

    sigma, lambda_ = sigma_points(x, P, alpha, beta, kappa)

    Wm = np.full(2*n+1, 1/(2*(n+lambda_)))
    Wc = np.full(2*n+1, 1/(2*(n+lambda_)))

    Wm[0] = lambda_/(n+lambda_)
    Wc[0] = lambda_/(n+lambda_) + (1-alpha**2+beta)

    sigma_pred = []

    for s in sigma:
        lat, vlat, lon, vlon = s
        sigma_pred.append([lat + vlat*dt_k, vlat, lon + vlon*dt_k, vlon])

    sigma_pred = np.array(sigma_pred)

    x_pred = np.sum(Wm[:,None] * sigma_pred, axis=0)

    P_pred = Q.copy()

    for i in range(2*n+1):
        diff = sigma_pred[i] - x_pred
        P_pred += Wc[i] * np.outer(diff, diff)

    Z = sigma_pred[:,[0,2]]
    z_pred = np.sum(Wm[:,None] * Z, axis=0)

    S = R.copy()

    for i in range(2*n+1):
        diff = Z[i] - z_pred
        S += Wc[i] * np.outer(diff, diff)

    Pxz = np.zeros((n,2))

    for i in range(2*n+1):
        dx = sigma_pred[i] - x_pred
        dz = Z[i] - z_pred
        Pxz += Wc[i] * np.outer(dx, dz)

    K = Pxz @ np.linalg.inv(S)

    z = positions[k]

    y = z - z_pred

    x = x_pred + K @ y
    P = P_pred - K @ S @ K.T

    ukf_estimates.append([x[0], x[2]])

    innovations.append(y)

    nis = y.T @ np.linalg.inv(S) @ y
    nis_values.append(nis)

ukf_estimates = np.array(ukf_estimates)
ukf_innovations = np.array(innovations)
ukf_nis = np.array(nis_values)

print("UKF complete")

## Innovation Magnitude Plot

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(np.linalg.norm(ukf_innovations, axis=1))

plt.title("UKF Innovation Magnitude (AIS Data)")
plt.xlabel("Time Step")
plt.ylabel("Innovation Norm")

plt.show()

## NIS Plot

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(ukf_nis)

plt.title("UKF Normalized Innovation Squared")
plt.xlabel("Time Step")
plt.ylabel("NIS")

plt.show()